# DPO - Direct Preference Optimization

This notebook shows a minimal, educational implementation of Direct Preference Optimization (DPO).

What we do:
- Build a tiny causal LM (policy model).
- Build synthetic preference pairs `(prompt, chosen, rejected)`.
- Compute sequence log-probabilities for chosen/rejected answers.
- Optimize the DPO objective against a frozen reference model.

Conceptual goal:
- In DPO, we skip that extra model and optimize preferences directly.
- The reward signal is implicit and comes from a log-probability ratio between policy and reference.


In [ ]:
# Setup
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(42)


## 1 Tiny LM (policy and reference backbone)
See EX04 as reference.

In [ ]:
class TinyLanguageModel(nn.Module):
    """Tiny model that maps token IDs -> logits [B, S, V]."""

    def __init__(self, vocab_size=50, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(num_embeddings=vocab_size, embedding_dim=hidden_dim)
        self.ff = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, vocab_size),
        )

    def forward(self, input_ids, padding_mask=None):
        x = self.embed(input_ids)          # [B, S] -> [B, S, H]
        logits = self.ff(x)                # [B, S, H] -> [B, S, V]
        return logits


## 2 Synthetic preference dataset

DPO needs pairwise preferences for the same prompt:
- `chosen`: preferred response
- `rejected`: less preferred response

(prompt, chosen_response, rejected_response)

In this toy dataset, preference is generated by a simple rule:
- Chosen responses sample larger token IDs.
- Rejected responses sample smaller token IDs.

-> In this example: We want to DPO the model to prefere larger token IDs

Data formatting detail:
- Inputs are concatenated as `[prompt | response]`.
- Labels on prompt tokens are set to `-100`, so loss only scores response tokens.

This mirrors instruction tuning setups where prompt/context is conditioning, and supervision is on generated answer tokens.

---

**For PikoGPT use HF Datasets like Anthropic/hh-rlhf or stanfordnlp/SHP**


In [ ]:
class ToyPreferenceDataset(Dataset):
    """
    Returns:
      - prompt_input_ids
      - chosen_input_ids
      - rejected_input_ids
      - chosen_labels
      - rejected_labels
      - chosen_padding_mask
      - rejected_padding_mask

    Convention:
      We concatenate [prompt | response].
      Loss is ONLY applied on response tokens.
      Prompt tokens in labels are set to -100.
    """

    def __init__(
        self,
        num_samples=64,
        prompt_len=4,
        response_len=6,
        vocab_size=50,
        pad_token_id=0,
    ):
        self.num_samples = num_samples
        self.prompt_len = prompt_len
        self.response_len = response_len
        self.vocab_size = vocab_size
        self.pad_token_id = pad_token_id

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Prompt tokens in [1, vocab_size-1]
        prompt = torch.randint(1, self.vocab_size, (self.prompt_len,), dtype=torch.long)

        # Synthetic rule for preference:
        # "chosen" responses use larger token IDs on average than "rejected" responses.
        # This creates a simple, learnable preference signal.
        chosen_response = torch.randint(
            low=self.vocab_size // 2,
            high=self.vocab_size,
            size=(self.response_len,),
            dtype=torch.long,
        )
        rejected_response = torch.randint(
            low=1,
            high=self.vocab_size // 2,
            size=(self.response_len,),
            dtype=torch.long,
        )

        chosen_input_ids = torch.cat([prompt, chosen_response], dim=0)
        rejected_input_ids = torch.cat([prompt, rejected_response], dim=0)

        chosen_padding_mask = torch.ones_like(chosen_input_ids, dtype=torch.bool)
        rejected_padding_mask = torch.ones_like(rejected_input_ids, dtype=torch.bool)

        # Labels: ignore prompt, score only response
        chosen_labels = chosen_input_ids.clone()
        rejected_labels = rejected_input_ids.clone()

        chosen_labels[:self.prompt_len] = -100
        rejected_labels[:self.prompt_len] = -100

        return {
            "chosen_input_ids": chosen_input_ids,
            "chosen_labels": chosen_labels,
            "chosen_padding_mask": chosen_padding_mask,
            "rejected_input_ids": rejected_input_ids,
            "rejected_labels": rejected_labels,
            "rejected_padding_mask": rejected_padding_mask,
        }

## 3 Sequence log-probability helper

DPO compares *sequence-level* likelihoods, not single-token predictions.

This helper:
- Applies `log_softmax` over vocabulary logits.
- Selects log-probabilities for target tokens with `gather`.
- Masks out ignored tokens (`-100`, i.e., prompt positions).
- Sums token log-probabilities to get one scalar per sequence.

Result:
- `log p_policy(y_chosen | x)`
- `log p_policy(y_rejected | x)`
- and the same for the reference model.


### Interpretation of the sequence_logprob_from_labels function: 
> How likely does the model think this entire response is, given the prompt?

> How much probability mass did the model assign to this whole answer?

In [ ]:
def sequence_logprob_from_labels(logits, labels):
    """
    Compute total log p(target tokens) for each sequence in the batch.

    logits: [B, S, V]
    labels: [B, S] with -100 on tokens to ignore

    Returns:
      seq_logprob: [B]
    """
    log_probs = F.log_softmax(logits, dim=-1)  # [B, S, V]

    safe_labels = labels.clone()
    safe_labels[safe_labels == -100] = 0  # temporary index for gather

    token_log_probs = log_probs.gather(dim=-1, index=safe_labels.unsqueeze(-1)).squeeze(-1)  # [B, S]

    mask = (labels != -100).float()
    seq_logprob = (token_log_probs * mask).sum(dim=-1)  # [B]

    return seq_logprob

## 4 DPO loss and implicit reward model

For each prompt `x`, DPO uses chosen/rejected responses `(y_w, y_l)` and optimizes:

`L = -log sigma(beta * [ (log pi(y_w|x) - log pi(y_l|x)) - (log pi_ref(y_w|x) - log pi_ref(y_l|x)) ])`

Interpretation:
- The policy should increase preference for chosen over rejected.
- The reference model regularizes this shift.
- `beta` controls sharpness / strength of the preference update.

Where is the reward model?
- There is no separate trainable reward network in this notebook.
- DPO induces an *implicit reward*:
  `r_hat(x, y) = beta * (log pi_theta(y|x) - log pi_ref(y|x))`
- Training effectively pushes `r_hat(x, y_w) > r_hat(x, y_l)`.


In [ ]:
def dpo_loss(
    policy_chosen_logp,
    policy_rejected_logp,
    ref_chosen_logp,
    ref_rejected_logp,
    beta=0.1,
):
    """
    DPO loss per example:
      -log sigmoid(beta * ((pi_chosen - pi_rejected) - (ref_chosen - ref_rejected)))

    Implicit reward view (no separate reward model is trained here):
      r_hat(x, y) = beta * (log pi_theta(y|x) - log pi_ref(y|x))
    and DPO encourages r_hat(chosen) > r_hat(rejected).
    """
    policy_logratios = policy_chosen_logp - policy_rejected_logp
    ref_logratios = ref_chosen_logp - ref_rejected_logp

    advantages = policy_logratios - ref_logratios
    losses = -F.logsigmoid(beta * advantages)

    implicit_reward_chosen = beta * (policy_chosen_logp - ref_chosen_logp)
    implicit_reward_rejected = beta * (policy_rejected_logp - ref_rejected_logp)

    return losses.mean(), {
        "policy_logratios": policy_logratios.detach().mean().item(),
        "ref_logratios": ref_logratios.detach().mean().item(),
        "advantages": advantages.detach().mean().item(),
        "implicit_reward_chosen": implicit_reward_chosen.detach().mean().item(),
        "implicit_reward_rejected": implicit_reward_rejected.detach().mean().item(),
        "implicit_reward_margin": (implicit_reward_chosen - implicit_reward_rejected).detach().mean().item(),
    }


Plot results

In [ ]:
# Plot function
def plot_training_history(history):
    """Plot key DPO training metrics across optimization steps."""
    steps = history["step"]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(steps, history["loss"])
    axes[0].set_title("DPO Loss")
    axes[0].set_xlabel("Step")
    axes[0].set_ylabel("Loss")
    axes[0].grid(alpha=0.3)

    axes[1].plot(steps, history["advantage"], label="advantage")
    axes[1].plot(steps, history["implicit_reward_margin"], label="implicit_reward_margin")
    axes[1].set_title("Preference Signal")
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Value")
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    axes[2].plot(steps, history["policy_logratio"], label="policy_logratio")
    axes[2].plot(steps, history["ref_logratio"], label="ref_logratio")
    axes[2].set_title("Policy vs Reference")
    axes[2].set_xlabel("Step")
    axes[2].set_ylabel("Log-ratio")
    axes[2].legend()
    axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

## 5 Training script

Per batch:
1. Run policy on chosen and rejected sequences.
2. Convert logits to sequence log-probabilities.
3. Run frozen reference model on the same sequences (no gradients).
4. Compute DPO loss from policy vs reference log-ratio differences.
5. Backprop only through the policy model.

Why freeze the reference model?
- It anchors behavior near the starting policy distribution.
- It prevents unconstrained preference chasing.
- It provides the baseline used by the implicit reward expression.

So, in code terms, the reward signal is represented by log-probability differences involving
`policy_model` and `reference_model`, not by a standalone `reward_model` object.


In [ ]:
def main():
    # hyperparameters
    vocab_size = 500
    hidden_dim = 32
    batch_size = 8
    num_epochs = 4
    lr = 1e-2
    beta = 0.5

    ## Create synthetic toy dataset
    dataset = ToyPreferenceDataset(
        num_samples=128,
        prompt_len=4,
        response_len=6,
        vocab_size=vocab_size,
        pad_token_id=0,
    )
    # batch multiple preference examples together
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # load the policy model -> the model we train
    policy_model = TinyLanguageModel(vocab_size=vocab_size, hidden_dim=hidden_dim)

    # Frozen reference model starts as a copy of the initial policy.
    # DPO uses policy-vs-reference log-prob differences as an implicit reward signal.
    reference_model = copy.deepcopy(policy_model)

    # freeze the reference model
    for p in reference_model.parameters():
        p.requires_grad = False
    reference_model.eval()

    # define adam as optimizer for the policy model for training
    optimizer = torch.optim.Adam(policy_model.parameters(), lr=lr)

    history = {
        "step": [],
        "epoch": [],
        "loss": [],
        "policy_logratio": [],
        "ref_logratio": [],
        "advantage": [],
        "implicit_reward_margin": [],
    }

    global_step = 0

    ## TRAINING LOOP ##
    for epoch in range(num_epochs):
        for batch_idx, batch in enumerate(loader, start=1):
            # unpack data
            chosen_input_ids = batch["chosen_input_ids"]              # [B, S]
            chosen_labels = batch["chosen_labels"]                    # [B, S]
            chosen_padding_mask = batch["chosen_padding_mask"]        # [B, S]
            rejected_input_ids = batch["rejected_input_ids"]          # [B, S]
            rejected_labels = batch["rejected_labels"]                # [B, S]
            rejected_padding_mask = batch["rejected_padding_mask"]    # [B, S]

            # 1. Score chosen/rejected under the trainable policy
            # two forward passes:
            # - one for the chosen continuation
            # - one for the rejected continuation
            # both with same prompt but different response
            policy_chosen_logits = policy_model(chosen_input_ids, chosen_padding_mask)
            policy_rejected_logits = policy_model(rejected_input_ids, rejected_padding_mask)

            # get total log-probability for each chosen/rejected sequence
            policy_chosen_logp = sequence_logprob_from_labels(policy_chosen_logits, chosen_labels)
            policy_rejected_logp = sequence_logprob_from_labels(policy_rejected_logits, rejected_labels)

            # 2. Score chosen/rejected under the FROZEN reference model
            with torch.no_grad():
                ref_chosen_logits = reference_model(chosen_input_ids, chosen_padding_mask)
                ref_rejected_logits = reference_model(rejected_input_ids, rejected_padding_mask)
                # get total log-probability for each chosen/rejected sequence
                ref_chosen_logp = sequence_logprob_from_labels(ref_chosen_logits, chosen_labels)
                ref_rejected_logp = sequence_logprob_from_labels(ref_rejected_logits, rejected_labels)

            # 3. Compute DPO-Loss for training
            loss, metrics = dpo_loss(
                policy_chosen_logp=policy_chosen_logp,
                policy_rejected_logp=policy_rejected_logp,
                ref_chosen_logp=ref_chosen_logp,
                ref_rejected_logp=ref_rejected_logp,
                beta=beta,
            )

            # 4. Training step
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            global_step += 1


            # append results to history for Plotting
            history["step"].append(global_step)
            history["epoch"].append(epoch + 1)
            history["loss"].append(loss.item())
            history["policy_logratio"].append(metrics["policy_logratios"])
            history["ref_logratio"].append(metrics["ref_logratios"])
            history["advantage"].append(metrics["advantages"])
            history["implicit_reward_margin"].append(metrics["implicit_reward_margin"])

            # print results per optimization step
            print(
                f"Step {global_step:03d} "
                f"(epoch={epoch+1:02d}, batch={batch_idx:02d}/{len(loader):02d}) | "
                f"loss={loss.item():.4f} | "
                f"policy_logratio={metrics['policy_logratios']:.4f} | "
                f"ref_logratio={metrics['ref_logratios']:.4f} | "
                f"advantage={metrics['advantages']:.4f} | "
                f"implicit_reward_margin={metrics['implicit_reward_margin']:.4f}"
            )

    print("\nTraining finished.")
    return history


if __name__ == "__main__":
    history = main()

In [ ]:
plot_training_history(history)